# Pytorch 2.0 介紹

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/taipeitechmmslab/MMSLAB-TF2/blob/master/Lab1.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/taipeitechmmslab/MMSLAB-TF2/blob/master/Lab1.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

Pytorch 的基本型態：

In [2]:
import torch
# 產生一個常數
c = torch.tensor(1)
# 產生一個變數 (設定 requires_grad=True 讓 PyTorch 知道它需要計算梯度)
# 注意：在 PyTorch 中，只有浮點數 (float) 等型別可以設定 requires_grad=True
v = torch.tensor(1.0, requires_grad=True)
print(c)
print(v)

tensor(1)
tensor(1., requires_grad=True)


零階張量稱為標量。

In [3]:
x = torch.tensor(4)
print(x)  # 顯示Tensor 常數資訊，Shape=()表示標量，dtype=int32 表示整數
print("{} 階Tensor".format(x.dim()))  # 顯示Tensor 的維度

tensor(4)
0 階Tensor


一階張量稱為向量。

In [4]:
x = torch.tensor([1, 2, 3, 4, 5, 6]) 
print("{}階Tensor ".format(x.dim()))    # 顯示Tensor的維度

1階Tensor 


二階張量稱為矩陣。

In [5]:
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("{}階Tensor ".format(x.dim()))    # 顯示Tensor的維度

2階Tensor 


---
## 動態圖


Pytorch 引入了「動態圖」動態圖模式，這個模式在Pytorch2.0為預設模式，不同與以往的靜態圖模式需要建立計算圖才能執行，動態圖模式一旦執行就會返回數值。這使Pytorch 更容易入門，也使研發更直觀。

**動態圖 的優點如下：**

- 立即返回數值，方便除錯。
- 無需 Session.run() 就可以把它們的值返回到 Python。
- 為自定義和高階梯度提供強大支援。
- 幾乎所有 Pytorch 運算都適用。


**Pytorch 1.x 和 Pytorch2.0比較：**
```
Pytorch 1.x code:
>>> a = tf.constant(1)
>>> print(a)
Tensor("Const_5:0", shape=(), dtype=int32)

>>> sess = tf.Session()
>>> print("a = {}".format(sess.run(a)))
a = 1


Pytorch 2.0 code:
>>> a = tf.constant(1)
>>> print(a)
tf.Tensor(1, shape=(), dtype=int32)
```

**Pytorch 基本運算：**

Import 必要套件

In [ ]:
import numpy as np
import torch

print("Eager Execution 是否啟動: {}".format(True))

定義常數 Tensor

In [ ]:
a = torch.tensor(3)
b = torch.tensor(4)
print("a = {}".format(a))
print("b = {}".format(b))

檢查資料型態

In [ ]:
print(a)
print(b)

基本的運算

In [ ]:
c = a + torch.tensor(b)
print("a + b = {}".format(c))
d = a * b
print("a * b = {}".format(d))

2D Tensor 的運算，在動態圖模式下可以混和 Tensor 和 Numpy 做運算

In [ ]:
a = torch.tensor([[1., 2.], [3., 4.]], dtype=torch.float32)
b = np.array([[1., 0.], [2., 3.]], dtype=np.float32)
print("a constant: {}D Tensor".format(a.dim()))

c = a + torch.tensor(b)
print("a + b = \n{}".format(c))
d = torch.matmul(a, torch.tensor(b))
print("a * b = \n{}".format(d))

輸出的結果為 Tensor 格式，我們可以將它轉為 Numpy 格式

In [ ]:
print(c)
print("NumpyArray:\n {}".format(c.detach().numpy()))

計算梯度

In [ ]:
w = torch.tensor([[1.0]], requires_grad=True)
loss = w * w
loss.backward()

grad = w.grad
print(grad)

---
## Keras

Pytorch2.0將Keras納為內建高階API，因此不必再而外安裝torch.nn套件，直接透過`torch.nn`指令使用。相較於`torch.nn`和`torch.nn`的差別，`torch.nn`更能全面支援tensorflow的指令與模式，例如支援Eager Exection, torch.utils.data, TPU訓練等等。

下面將會介紹兩種最常使用的網路搭建方法：
- nn.Sequential (序列模型)
- Function API (函數式模型)

### Import 必要套件

In [ ]:
import torch
import torch.nn as nn
from IPython.display import Image

### nn.Sequential

nn.Sequential有兩種方法搭建，下面為分類問題範例：輸入為28x28 (拉平為784的一維向量)的影像，輸出為10 (分為十個類別)。

**Ex1：**

In [ ]:
# 產生網絡模型
model = nn.Sequential(
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
    nn.Softmax(dim=1)
)

**Ex2：**

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
    nn.Softmax(dim=1)
)

顯示出剛剛搭建的網路

In [ ]:
!pip install torchsummary
from torchsummary import summary
# 使用 torchsummary 顯示網路架構與參數數量
summary(model, (784,))

---
## torch.utils.data

### 基本操作

1.`torch.utils.data.Dataset.from_tensors`

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 產生包含單一高維張量資料集
tensor = torch.tensor([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]])
dataset = TensorDataset(tensor)
print(dataset)

2.`torch.utils.data.Dataset.from_tensor_slices`

In [ ]:
x_tensor = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
x_data = TensorDataset(x_tensor)
print(x_data)

y_tensor = torch.tensor([0, 2, 4, 6, 8, 10, 12, 14, 16, 18])
y_data = TensorDataset(y_tensor)
print(y_data)

3.`for loop`讀取數據：

In [ ]:
for data in dataset:
    print(data)

In [ ]:
for data1, data2 in zip(x_data, y_data):
    print('x: {}, y: {}'.format(data1[0], data2[0]))

3.`take`：讀取資料

In [ ]:
for data in dataset:
    print(data)

In [ ]:
for i in range(min(5, len(x_data))):
    print('x: {}, y: {}'.format(x_data[i][0], y_data[i][0]))

In [ ]:
for i in range(min(12, len(x_data))):
    print('x: {}, y: {}'.format(x_data[i][0], y_data[i][0]))

5.`torch.utils.data.Dataset.zip`

In [ ]:
dataset = TensorDataset(x_tensor, y_tensor)
print(dataset)

6.`map`：可以使用map來轉換數據

In [ ]:
mapped_data = [x * 2 for x in torch.arange(10)]
print(mapped_data)

7.命名：可以字典方式為elements的組件命名

In [ ]:
mapped_data = [x * 2 for x in torch.arange(10)]
print(mapped_data)

In [ ]:
for i in range(min(10, len(dict_dataset))):
    print('x: {}, y: {}'.format(dict_dataset[i]['x'], dict_dataset[i]['y']))

8.設定每batch讀取的數量

In [ ]:
from torch.utils.data import DataLoader
dataloader = DataLoader(dict_dataset, batch_size=2)

for i, data in enumerate(dataloader):
    if i >= 5: break
    print('x: {}, y: {}'.format(data['x'].numpy(), data['y'].numpy()))

9.`shuffle`：dataset資料 會被載入buffer中，並從buffer中隨機選取資料出來，取出資料產生的空位會從新的數據替補。而`buffer_size`是設定buffer大小，最好的設定是**大於**或**等於**整個dataset資料個個數。

In [ ]:
shuffle_dataloader = DataLoader(dict_dataset, batch_size=2, shuffle=True)
for i, data in enumerate(shuffle_dataloader):
    if i >= 5: break
    print('x: {}, y: {}'.format(data['x'].numpy(), data['y'].numpy()))

10.`repeat`：當dataset的資料讀取完後就讀會取不到資料，透過設定`repeat(n)`可以重複讀取dataset n次。

In [ ]:
for i in range(min(10, len(dict_dataset))):
    print('x: {}, y: {}'.format(dict_dataset[i]['x'], dict_dataset[i]['y']))